# SAM3 smoke tests

Constructor-argument reference for the two SAM3 entry points, using the bundled test media (not dolphin footage):

1. **Image path** — text-prompted detection on `dogs.jpg`.
2. **Video path** — SAM 3.1 multiplex video predictor on `dogs_playing.mp4`.

If flash-attn-3 (`use_fa3=True`) fails to import or launch on the RTX 4090 (Ada/sm_89), rebuild with `use_fa3=False` — there is a verified SDPA fallback.

In [ ]:
import torch

from fish_segmentation.paths import find_repo_root, load_env

load_env()
ROOT = find_repo_root()

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"using device: {device}")

if device.type == "cuda":
    # use bfloat16 for the entire notebook
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    # turn on tfloat32 for Ampere GPUs
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

## Image example

In [ ]:
from PIL import Image

image_path = ROOT / "data" / "test_media" / "dogs.jpg"
image = Image.open(image_path)
image

In [ ]:
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.model_builder import build_sam3_image_model

model = build_sam3_image_model()
processor = Sam3Processor(model)

IMAGE_TEXT_PROMPT = "dogs"

inference_state = processor.set_image(image)
output = processor.set_text_prompt(state=inference_state, prompt=IMAGE_TEXT_PROMPT)
masks, boxes, scores = output["masks"], output["boxes"], output["scores"]

In [ ]:
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np

masks_np = masks.cpu().float().numpy()
boxes_np = boxes.cpu().float().numpy()
scores_np = scores.cpu().float().numpy()

plt.figure(figsize=(10, 10))
plt.imshow(image)
ax = plt.gca()

for i in range(len(masks_np)):
    mask = masks_np[i]
    if len(mask.shape) == 3:
        mask = mask[0]

    color = np.concatenate([np.random.random(3), np.array([0.5])], axis=0)
    h, w = mask.shape
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

    box = boxes_np[i]
    x_min, y_min, x_max, y_max = box
    rect = patches.Rectangle(
        (x_min, y_min),
        x_max - x_min,
        y_max - y_min,
        linewidth=2,
        edgecolor='red',
        facecolor='none'
    )
    ax.add_patch(rect)

    score = scores_np[i]
    ax.text(
        x_min, y_min - 5,
        f"Score: {score:.2f}",
        color='white',
        fontsize=10,
        weight='bold',
        bbox=dict(facecolor='red', edgecolor='red', alpha=0.8, pad=1)
    )

plt.axis('off')
plt.tight_layout()
plt.show()

## Video example

In [ ]:
import gc
import matplotlib.pyplot as plt

from fish_segmentation.sam3_utils import add_text_prompt, propagate_in_video
from fish_segmentation.video_io import load_all_frames_rgb
from sam3.model_builder import build_sam3_multiplex_video_predictor
from sam3.visualization_utils import (
    prepare_masks_for_visualization,
    visualize_formatted_frame_output,
)

plt.rcParams["axes.titlesize"] = 12
plt.rcParams["figure.titlesize"] = 12

predictor = build_sam3_multiplex_video_predictor(
    checkpoint_path=None,
    bpe_path=None,
    max_num_objects=16,
    multiplex_count=16,
    use_fa3=True,
    use_rope_real=True,
    compile=False,
    warm_up=False,
    session_expiration_sec=1200,
    default_output_prob_thresh=0.5,
    async_loading_frames=True,
)

video_path = ROOT / "data" / "test_media" / "dogs_playing.mp4"
video_frames_for_vis = load_all_frames_rgb(str(video_path))

In [ ]:
response = predictor.handle_request(
    request=dict(
        type="start_session",
        resource_path=str(video_path),
        offload_state_to_cpu=True,
    )
)
session_id = response["session_id"]

In [ ]:
# required before switching to a different text prompt in the same session
_ = predictor.handle_request(
    request=dict(
        type="reset_session",
        session_id=session_id,
    )
)

In [ ]:
prompt_text_str = "dog"
frame_idx = 0
out = add_text_prompt(predictor, session_id, frame_idx, prompt_text_str)

plt.close("all")
visualize_formatted_frame_output(
    frame_idx,
    video_frames_for_vis,
    outputs_list=[prepare_masks_for_visualization({frame_idx: out})],
    titles=["SAM 3.1 Dense Tracking outputs"],
    figsize=(6, 4),
)

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
outputs_per_frame = propagate_in_video(predictor, session_id)
outputs_per_frame = prepare_masks_for_visualization(outputs_per_frame)

vis_frame_stride = 60
plt.close("all")
for frame_idx in range(0, len(outputs_per_frame), vis_frame_stride):
    visualize_formatted_frame_output(
        frame_idx,
        video_frames_for_vis,
        outputs_list=[outputs_per_frame],
        titles=["SAM 3.1 Dense Tracking outputs"],
        figsize=(6, 4),
    )

In [ ]:
_ = predictor.handle_request(
    request=dict(
        type="close_session",
        session_id=session_id,
    )
)
predictor.shutdown()